# Kaggle S6E10 — Predicting Airline Satisfaction

End-to-end pipeline notebook. Reproduces the 5-fold cross-validated ensemble
that currently sits at **OOF AUC = 0.960865** (blend) and **0.960757** (single).

Best public LB target: ≥ 0.9605 (goal), 0.9610 (stretch), 0.9620 (ceiling).

**Inputs**
- `playground-series-dataCSV/train.csv` (699,635 rows, 24 cols, binary `satisfaction`)
- `playground-series-dataCSV/test.csv` (299,844 rows, 23 cols)
- `playground-series-dataCSV/sample_submission.csv` (id, satisfaction)

**Outputs**
- `submissions/best.csv` — 5-OOF stack blend, current best
- `submissions/best_single.csv` — `lgbm_full_g` champion
- `oof/*.npy` — OOF predictions for each base learner
- `preds/*.npy` — test predictions for each base learner

**Hardware (local)**
- Apple Silicon, 8 CPU cores, 16 GB RAM, no GPU
- Total runtime: ~2 hours end-to-end (dominated by Optuna search + 5-fold refits)

## 0. Setup

In [ ]:
import os, sys, time, json, glob
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

ROOT = os.path.dirname(os.path.abspath('pipeline.ipynb')) if '__file__' not in globals() else os.path.dirname(os.path.abspath(__file__))
DATA_DIR = os.path.join(ROOT, 'playground-series-dataCSV')
OOF_DIR   = os.path.join(ROOT, 'oof')
PRED_DIR  = os.path.join(ROOT, 'preds')
SUB_DIR   = os.path.join(ROOT, 'submissions')
for d in (OOF_DIR, PRED_DIR, SUB_DIR):
    os.makedirs(d, exist_ok=True)

CAT_COLS = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
NUM_COLS = ['Age', 'Flight Distance',
            'Departure Delay in Minutes', 'Arrival Delay in Minutes']
N_FOLDS = 5
SEED = 42
GATE = 6.43e-05  # = 2 * stdev of OOF AUC across 3 seeds on lgbm_raw

In [ ]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
print('lgb', lgb.__version__, '| xgb', xgb.__version__, '| optuna', optuna.__version__)

## 1. Data loading

Reads `train.csv` and `test.csv`, strips column whitespace, and creates a
`__y__` column (binary 0/1) on the train frame.

In [ ]:
def load_train_test():
    tr = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
    te = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
    tr.columns = [c.strip() for c in tr.columns]
    te.columns = [c.strip() for c in te.columns]
    if tr['satisfaction'].dtype == bool:
        tr['__y__'] = tr['satisfaction'].astype(np.int8)
    else:
        m = {'satisfied': 1, 'neutral or dissatisfied': 0}
        tr['__y__'] = tr['satisfaction'].astype(str).str.strip().map(m).astype(np.int8)
    return tr, te

def rating_cols(tr):
    DROP = ['id', 'satisfaction', '__y__']
    RAW = [c for c in tr.columns if c not in DROP]
    return [c for c in RAW if c not in CAT_COLS + NUM_COLS]

tr, te = load_train_test()
print('train', tr.shape, '| test', te.shape, '| ratings', len(rating_cols(tr)))

## 2. Folds (frozen)

5-fold StratifiedKFold with `shuffle=True, random_state=42`. Saved to
`folds.csv` so every experiment reuses the exact same split.

In [ ]:
FOLD_FILE = os.path.join(ROOT, 'folds.csv')

def build_folds():
    y = tr['__y__'].values
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold = np.zeros(len(tr), dtype=np.int8)
    for k, (_, vi) in enumerate(skf.split(tr, y)):
        fold[vi] = k
    pd.DataFrame({'id': tr['id'].values, 'fold': fold}).to_csv(FOLD_FILE, index=False)
    return fold

if not os.path.exists(FOLD_FILE):
    folds = build_folds()
else:
    folds = pd.read_csv(FOLD_FILE).set_index('id').loc[tr['id'].values, 'fold'].astype(np.int8).values
print('folds:', np.bincount(folds))

## 3. Feature builders

Three composable feature groups:

1. **Route aggregates** (`exp_B_route.build_route_features`): per-Flight-Distance
   count, mean/std of each rating + Age, and per-source category share.
2. **Small interactions** (`exp_D_small._add_small_features`): log delays,
   delay diff, missing flag, rating aggregates, type×class×customer string.
3. **Target encoding** (`exp_C_te._te_4way_kfold`, `_te_flight_dist_kfold`):
   4-way smoothed TE on (Class, Type, Customer, FlightDist) and 1-way TE on
   Flight Distance, both fold-safe.

All features are label-free (they don't look at `__y__` of the same row).

In [ ]:
def _route_aggregates(df, route_col, agg_cols):
    g = df.groupby(route_col, observed=True)
    out = pd.DataFrame({route_col: g.size().index})
    out['route_count'] = g.size().values
    for c in agg_cols:
        out[f'route_{c}_mean'] = g[c].mean().values
        out[f'route_{c}_std']  = g[c].std().fillna(0).values
    return out

def _per_source_counts(df, route_col, src_col='__src__'):
    g = df.groupby([route_col, src_col], observed=True).size().unstack(fill_value=0)
    g.columns = [f'route_src{int(c)}_count' for c in g.columns]
    return g.reset_index()

def _per_route_cat_shares(df, route_col, cat_cols):
    out = pd.DataFrame({route_col: df[route_col].drop_duplicates().values})
    for c in cat_cols:
        dummies = pd.get_dummies(df[c].astype(str), prefix=c).astype('int8')
        dummies[route_col] = df[route_col].values
        agg = dummies.groupby(route_col, observed=True).mean()
        agg.columns = [f'route_share_{col}' for col in agg.columns]
        out = out.merge(agg.reset_index(), on=route_col, how='left')
    return out

def build_route_features(tr_, te_):
    RATING = rating_cols(tr_)
    src_tr = tr_[['Flight Distance']].copy()
    src_tr['__src__'] = 0
    src_te = te_[['Flight Distance']].copy()
    src_te['__src__'] = 1
    all_ = pd.concat([src_tr.assign(**{c: tr_[c] for c in RATING + ['Age']}),
                      src_te.assign(**{c: te_[c] for c in RATING + ['Age']})],
                     ignore_index=True)
    agg = _route_aggregates(all_, 'Flight Distance', RATING + ['Age'])
    src_c = _per_source_counts(all_, 'Flight Distance')
    cat_share = _per_route_cat_shares(all_, 'Flight Distance', CAT_COLS)
    out_tr = tr_.merge(agg, on='Flight Distance', how='left')
    out_tr = out_tr.merge(src_c, on='Flight Distance', how='left')
    out_tr = out_tr.merge(cat_share, on='Flight Distance', how='left')
    out_te = te_.merge(agg, on='Flight Distance', how='left')
    out_te = out_te.merge(src_c, on='Flight Distance', how='left')
    out_te = out_te.merge(cat_share, on='Flight Distance', how='left')
    return out_tr, out_te

def _add_small_features(df):
    RATING = rating_cols(df) if 'satisfaction' in df.columns else \
             [c for c in df.columns if c not in ['id','satisfaction','__y__']+CAT_COLS+NUM_COLS]
    out = df.copy()
    dep = out['Departure Delay in Minutes'].astype('float32')
    arr = out['Arrival Delay in Minutes'].astype('float32')
    out['log_dep_delay'] = np.log1p(dep.fillna(0)).astype('float32')
    out['log_arr_delay'] = np.log1p(arr.fillna(0)).astype('float32')
    out['total_delay']   = (dep.fillna(0) + arr.fillna(0)).astype('float32')
    out['max_delay']     = np.maximum(dep.fillna(0), arr.fillna(0)).astype('float32')
    out['min_delay']     = np.minimum(dep.fillna(0), arr.fillna(0)).astype('float32')
    out['delay_diff']    = (dep.fillna(0) - arr.fillna(0)).astype('float32')
    out['has_dep_delay'] = (dep.fillna(0) > 0).astype('int8')
    out['has_arr_delay'] = (arr.fillna(0) > 0).astype('int8')
    out['on_time']       = ((dep.fillna(0) == 0) & (arr.fillna(0) == 0)).astype('int8')
    out['arr_delay_missing'] = arr.isna().astype('int8')
    rvals = out[RATING].astype('float32')
    out['rating_mean'] = rvals.mean(axis=1).astype('float32')
    out['rating_std']  = rvals.std(axis=1).astype('float32')
    out['rating_min']  = rvals.min(axis=1).astype('float32')
    out['rating_max']  = rvals.max(axis=1).astype('float32')
    out['rating_range']= (out['rating_max']-out['rating_min']).astype('float32')
    out['n_zero_ratings'] = (rvals == 0).sum(axis=1).astype('int8')
    out['flat_rater'] = (out['rating_std'] < 0.5).astype('int8')
    out['low_rating_frac']  = (rvals <= 2).mean(axis=1).astype('float32')
    out['high_rating_frac'] = (rvals >= 4).mean(axis=1).astype('float32')
    if {'Type of Travel','Class','Customer Type'}.issubset(out.columns):
        out['type_x_class_x_customer'] = (
            out['Type of Travel'].astype(str) + '|' +
            out['Class'].astype(str) + '|' +
            out['Customer Type'].astype(str)
        ).astype('category')
    return out

def _te_4way_kfold(df, folds_, PRIOR, SMOOTH=20, cols=('Class','Type of Travel','Customer Type','Flight Distance')):
    """Fold-safe smoothed target encoding on a 4-way key."""
    keys = df[list(cols[0:3])].astype(str).agg('|'.join, axis=1) + '|' + df[cols[3]].astype(str)
    n = len(df)
    oof = np.zeros(n, dtype=np.float32)
    for k in range(N_FOLDS):
        tr_m = folds_ != k
        te_keys = keys[tr_m]
        y_tr = df['__y__'].values[tr_m]
        agg = pd.DataFrame({'k': te_keys, 'y': y_tr}).groupby('k')['y'].agg(['sum','count'])
        agg['enc'] = (agg['sum'] + SMOOTH * PRIOR) / (agg['count'] + SMOOTH)
        m = ~tr_m
        oof[m] = keys[m].map(agg['enc']).fillna(PRIOR).astype(np.float32).values
    full = pd.DataFrame({'k': keys, 'y': df['__y__'].values}).groupby('k')['y'].agg(['sum','count'])
    full['enc'] = (full['sum'] + SMOOTH * PRIOR) / (full['count'] + SMOOTH)
    return oof, full['enc']

def _te_flight_dist_kfold(df, folds_, PRIOR, SMOOTH=20):
    g_tr = df['Flight Distance'].astype(str)
    n = len(df)
    oof = np.zeros(n, dtype=np.float32)
    for k in range(N_FOLDS):
        tr_m = folds_ != k
        keys = g_tr[tr_m]
        y_tr = df['__y__'].values[tr_m]
        agg = pd.DataFrame({'k': keys, 'y': y_tr}).groupby('k')['y'].agg(['sum','count'])
        agg['enc'] = (agg['sum'] + SMOOTH * PRIOR) / (agg['count'] + SMOOTH)
        m = ~tr_m
        oof[m] = g_tr[m].map(agg['enc']).fillna(PRIOR).astype(np.float32).values
    full = pd.DataFrame({'k': g_tr, 'y': df['__y__'].values}).groupby('k')['y'].agg(['sum','count'])
    full['enc'] = (full['sum'] + SMOOTH * PRIOR) / (full['count'] + SMOOTH)
    return oof, full['enc']

def add_te_features(tr_enr, te_enr, folds_, PRIOR):
    """Add 4-way TE + 1-way FD TE to both train and test."""
    tr4_oof, tr4_full = _te_4way_kfold(tr_enr, folds_, PRIOR)
    te4 = te_enr.assign(**{c: te_enr[c] for c in ['Class','Type of Travel','Customer Type','Flight Distance']})
    te4_keys = (te4['Class'].astype(str)+'|'+te4['Type of Travel'].astype(str)+'|'+te4['Customer Type'].astype(str)+'|'+te4['Flight Distance'].astype(str))
    tr4 = tr_enr.assign(**{c: tr_enr[c] for c in ['Class','Type of Travel','Customer Type','Flight Distance']})
    tr4_keys = (tr4['Class'].astype(str)+'|'+tr4['Type of Travel'].astype(str)+'|'+tr4['Customer Type'].astype(str)+'|'+tr4['Flight Distance'].astype(str))
    out_tr = tr_enr.copy()
    out_tr['te_4way'] = tr4_oof
    out_te = te_enr.copy()
    out_te['te_4way'] = te4_keys.map(tr4_full).fillna(PRIOR).astype(np.float32).values
    trfd_oof, trfd_full = _te_flight_dist_kfold(tr_enr, folds_, PRIOR)
    out_tr['te_flight_dist'] = trfd_oof
    out_te['te_flight_dist'] = te_enr['Flight Distance'].astype(str).map(trfd_full).fillna(PRIOR).astype(np.float32).values
    return out_tr, out_te

def make_aligned_categoricals(*dfs):
    out = []
    for df in dfs:
        d = df.copy()
        for c in CAT_COLS:
            if c in d.columns:
                d[c] = d[c].astype('category')
        out.append(d)
    cats = {}
    for c in CAT_COLS:
        seen = [d[c].astype('category') for d in out if c in d.columns]
        if seen:
            cats[c] = pd.api.types.union_categoricals(seen).categories
    for d in out:
        for c, cc in cats.items():
            if c in d.columns:
                d[c] = d[c].astype(pd.CategoricalDtype(categories=cc))
    return out

print('Feature builders defined.')

## 4. STEP 0 — Sanity, noise, adversarial validation

Goal: ground the project on a plain LGBM (~0.9588) and set the
GATE = 2 × stdev(OOF AUC across 3 seeds).

In [ ]:
def lgbm_5fold(Xtr, y, Xte, folds_, seed=SEED, params=None, num_boost=4000, es=200):
    cat_cols = [c for c in Xtr.columns if str(Xtr[c].dtype) == 'category']
    if params is None:
        params = dict(objective='binary', metric='auc', learning_rate=0.05,
                      num_leaves=63, min_data_in_leaf=50, feature_fraction=0.85,
                      bagging_fraction=0.85, bagging_freq=1,
                      lambda_l1=0.1, lambda_l2=0.1,
                      verbosity=-1, seed=seed, num_threads=8, max_bin=255)
    else:
        params = {**params, 'verbosity': -1, 'num_threads': 8, 'seed': seed}
    oof = np.zeros(len(Xtr), dtype=np.float64)
    test_pred = np.zeros(len(Xte), dtype=np.float64)
    for k in range(N_FOLDS):
        tr_m = folds_ != k; va_m = folds_ == k
        dtr = lgb.Dataset(Xtr.iloc[tr_m], y[tr_m],
                          categorical_feature=cat_cols, free_raw_data=False)
        dva = lgb.Dataset(Xtr.iloc[va_m], y[va_m],
                          categorical_feature=cat_cols, reference=dtr,
                          free_raw_data=False)
        m = lgb.train(params, dtr, num_boost_round=num_boost, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(es, verbose=False),
                                 lgb.log_evaluation(0)])
        oof[va_m] = m.predict(Xtr.iloc[va_m], num_iteration=m.best_iteration)
        test_pred += m.predict(Xte, num_iteration=m.best_iteration) / N_FOLDS
    return oof, test_pred

# Sanity: raw features only, 1 seed
feats_raw = [c for c in tr.columns if c not in ('id','satisfaction','__y__')]
Xtr_raw = tr[feats_raw].copy()
Xte_raw = te[feats_raw].copy()
for c in CAT_COLS:
    if c in Xtr_raw.columns:
        Xtr_raw[c] = Xtr_raw[c].astype('category')
        Xte_raw[c] = Xte_raw[c].astype('category')
Xtr_raw, Xte_raw = make_aligned_categoricals(Xtr_raw, Xte_raw)
y = tr['__y__'].values
oof_raw, _ = lgbm_5fold(Xtr_raw, y, Xte_raw, folds, seed=SEED)
print(f'lgbm_raw OOF AUC: {roc_auc_score(y, oof_raw):.6f}  (expected ~0.9588)')
np.save(os.path.join(OOF_DIR, 'lgbm_raw.npy'), oof_raw.astype(np.float32))

In [ ]:
# Noise check: 3 seeds → GATE
aucs = []
for s in (42, 53, 64):
    o, _ = lgbm_5fold(Xtr_raw, y, Xte_raw, folds, seed=s)
    aucs.append(roc_auc_score(y, o))
print('aucs:', aucs, 'stdev:', np.std(aucs))
GATE = max(5e-5, 2 * np.std(aucs))
print('GATE =', GATE)

In [ ]:
# Adversarial validation: train vs test
all_X = pd.concat([Xtr_raw.assign(__src__=0), Xte_raw.assign(__src__=1)], ignore_index=True)
y_adv = all_X['__src__'].values
X_adv = all_X.drop(columns=['__src__'])
oof_adv, _ = lgbm_5fold(X_adv, y_adv, X_adv.head(100), folds[:len(X_adv)], seed=SEED)
print(f'Adversarial AUC: {roc_auc_score(y_adv, oof_adv):.4f}  (close to 0.5 = train/test indistinguishable)')
print('JSON: gate =', GATE, '| adv_auc =', round(roc_auc_score(y_adv, oof_adv), 4))

## 5. Champion pipeline

Builds the full feature set (route + small interactions + TE) and trains
the **3-seed-averaged LGBM at Optuna params** (`lgbm_full_g`, the current
champion with OOF AUC = 0.960757).

In [ ]:
CHAMPION_PARAMS = dict(
    learning_rate=0.020340260398623772,
    num_leaves=138,
    min_data_in_leaf=68,
    feature_fraction=0.7483052441171618,
    bagging_fraction=0.8870395218311953,
    bagging_freq=1,
    lambda_l1=4.806655024757077,
    lambda_l2=0.06748507222716818,
    max_bin=127,
)
SEEDS_G = [42, 53, 64]

def build_full_features():
    tr_enr, te_enr = build_route_features(tr, te)
    tr_enr = _add_small_features(tr_enr)
    te_enr = _add_small_features(te_enr)
    PRIOR = float(tr['__y__'].mean())
    tr_enr, te_enr = add_te_features(tr_enr, te_enr, folds, PRIOR)
    feats = [c for c in tr_enr.columns if c not in ('id','satisfaction','__y__','__src__')]
    Xtr = tr_enr[feats].copy()
    Xte = te_enr[feats].copy()
    for c in CAT_COLS:
        if c in Xtr.columns:
            Xtr[c] = Xtr[c].astype('category')
            Xte[c] = Xte[c].astype('category')
    Xtr, Xte = make_aligned_categoricals(Xtr, Xte)
    return Xtr, Xte, feats

Xtr, Xte, feats = build_full_features()
print('Feature matrix:', Xtr.shape, '| test', Xte.shape)

In [ ]:
def lgbm_seeds_5fold(Xtr, y, Xte, folds_, seeds, params, num_boost=6000, es=300):
    cat_cols = [c for c in Xtr.columns if str(Xtr[c].dtype) == 'category']
    oof = np.zeros(len(Xtr), dtype=np.float64)
    test_pred = np.zeros(len(Xte), dtype=np.float64)
    fold_aucs = []
    for k in range(N_FOLDS):
        tr_m = folds_ != k; va_m = folds_ == k
        oof_k = np.zeros(int(va_m.sum()))
        test_k = np.zeros(len(Xte))
        for s in seeds:
            p = {**params, 'objective':'binary', 'metric':'auc',
                 'verbosity':-1, 'num_threads':8, 'seed':s}
            dtr = lgb.Dataset(Xtr.iloc[tr_m], y[tr_m], categorical_feature=cat_cols, free_raw_data=False)
            dva = lgb.Dataset(Xtr.iloc[va_m], y[va_m], categorical_feature=cat_cols, reference=dtr, free_raw_data=False)
            m = lgb.train(p, dtr, num_boost_round=num_boost, valid_sets=[dva],
                          callbacks=[lgb.early_stopping(es, verbose=False), lgb.log_evaluation(0)])
            oof_k += m.predict(Xtr.iloc[va_m], num_iteration=m.best_iteration) / len(seeds)
            test_k += m.predict(Xte, num_iteration=m.best_iteration) / len(seeds)
        oof[va_m] = oof_k
        test_pred += test_k / N_FOLDS
        fold_aucs.append(roc_auc_score(y[va_m], oof[va_m]))
    return oof, test_pred, fold_aucs

t0 = time.time()
oof_g, test_g, fold_aucs = lgbm_seeds_5fold(Xtr, y, Xte, folds, SEEDS_G, CHAMPION_PARAMS)
print(f'lgbm_full_g OOF AUC: {roc_auc_score(y, oof_g):.6f}  per-fold {np.round(fold_aucs, 4).tolist()}  ({time.time()-t0:.0f}s)')
np.save(os.path.join(OOF_DIR, 'lgbm_full_g.npy'), oof_g.astype(np.float32))
np.save(os.path.join(PRED_DIR, 'lgbm_full_g.npy'), test_g.astype(np.float32))

## 6. CatBoost (depth=4) — BLEND MEMBER

CatBoost brings ordered-boosting diversity. Slower than LGBM on CPU;
depth=4 is a sensible compromise.

In [ ]:
def cat_5fold(Xtr, y, Xte, folds_, depth=4, lr=0.05, iters=2000, seed=SEED):
    cat_cols = [c for c in Xtr.columns if c in CAT_COLS or str(Xtr[c].dtype) == 'category']
    for c in cat_cols:
        Xtr[c] = Xtr[c].astype(str).fillna('nan')
        Xte[c] = Xte[c].astype(str).fillna('nan')
    feats = list(Xtr.columns)
    Xtr = Xtr[feats]
    Xte = Xte[feats]
    cat_idx = [feats.index(c) for c in cat_cols if c in feats]
    oof = np.zeros(len(Xtr), dtype=np.float64)
    test_pred = np.zeros(len(Xte), dtype=np.float64)
    fold_aucs = []
    for k in range(N_FOLDS):
        tr_m = folds_ != k; va_m = folds_ == k
        train_pool = Pool(Xtr.iloc[tr_m], y[tr_m], cat_features=cat_idx)
        val_pool = Pool(Xtr.iloc[va_m], y[va_m], cat_features=cat_idx)
        m = CatBoostClassifier(iterations=iters, learning_rate=lr, depth=depth,
                               l2_leaf_reg=3.0, random_seed=seed, verbose=0,
                               eval_metric='AUC', loss_function='Logloss',
                               early_stopping_rounds=200, task_type='CPU', thread_count=8)
        m.fit(train_pool, eval_set=val_pool, use_best_model=True)
        oof[va_m] = m.predict_proba(val_pool)[:, 1]
        test_pred += m.predict_proba(Pool(Xte, cat_features=cat_idx))[:, 1] / N_FOLDS
        fold_aucs.append(roc_auc_score(y[va_m], oof[va_m]))
    return oof, test_pred, fold_aucs

Xtr_cat = Xtr.copy(); Xte_cat = Xte.copy()
t0 = time.time()
oof_cat, test_cat, fold_aucs = cat_5fold(Xtr_cat, y, Xte_cat, folds)
print(f'cat_d4 OOF AUC: {roc_auc_score(y, oof_cat):.6f}  per-fold {np.round(fold_aucs, 4).tolist()}  ({time.time()-t0:.0f}s)')
np.save(os.path.join(OOF_DIR, 'cat_d4.npy'), oof_cat.astype(np.float32))
np.save(os.path.join(PRED_DIR, 'cat_d4.npy'), test_cat.astype(np.float32))

## 7. Other base learners (BLEND MEMBERS)

Three more LGBM variants: raw features, route-only, route + TE, route + small.
Each is slightly worse on its own than the champion, but they add diversity
to the stacker.

In [ ]:
CHAMPION_BASE_PARAMS = dict(objective='binary', metric='auc', learning_rate=0.05,
                            num_leaves=63, min_data_in_leaf=50, feature_fraction=0.85,
                            bagging_fraction=0.85, bagging_freq=1,
                            lambda_l1=0.1, lambda_l2=0.1, max_bin=255)

# 7a. lgbm_full — route features only
def build_route_only():
    tr_enr, te_enr = build_route_features(tr, te)
    feats = [c for c in tr_enr.columns if c not in ('id','satisfaction','__y__','__src__')]
    Xtr = tr_enr[feats].copy()
    Xte = te_enr[feats].copy()
    for c in CAT_COLS:
        if c in Xtr.columns:
            Xtr[c] = Xtr[c].astype('category')
            Xte[c] = Xte[c].astype('category')
    Xtr, Xte = make_aligned_categoricals(Xtr, Xte)
    return Xtr, Xte

Xtr_r, Xte_r = build_route_only()
oof_r, test_r = lgbm_5fold(Xtr_r, y, Xte_r, folds, seed=SEED, params=CHAMPION_BASE_PARAMS)
print(f'lgbm_full OOF AUC: {roc_auc_score(y, oof_r):.6f}')
np.save(os.path.join(OOF_DIR, 'lgbm_full.npy'), oof_r.astype(np.float32))
np.save(os.path.join(PRED_DIR, 'lgbm_full.npy'), test_r.astype(np.float32))

In [ ]:
# 7b. lgbm_full_te — route + TE
def build_route_te():
    tr_enr, te_enr = build_route_features(tr, te)
    PRIOR = float(tr['__y__'].mean())
    tr_enr, te_enr = add_te_features(tr_enr, te_enr, folds, PRIOR)
    feats = [c for c in tr_enr.columns if c not in ('id','satisfaction','__y__','__src__')]
    Xtr = tr_enr[feats].copy()
    Xte = te_enr[feats].copy()
    for c in CAT_COLS:
        if c in Xtr.columns:
            Xtr[c] = Xtr[c].astype('category')
            Xte[c] = Xte[c].astype('category')
    Xtr, Xte = make_aligned_categoricals(Xtr, Xte)
    return Xtr, Xte

Xtr_te, Xte_te = build_route_te()
oof_te, test_te = lgbm_5fold(Xtr_te, y, Xte_te, folds, seed=SEED, params=CHAMPION_BASE_PARAMS)
print(f'lgbm_full_te OOF AUC: {roc_auc_score(y, oof_te):.6f}')
np.save(os.path.join(OOF_DIR, 'lgbm_full_te.npy'), oof_te.astype(np.float32))
np.save(os.path.join(PRED_DIR, 'lgbm_full_te.npy'), test_te.astype(np.float32))

In [ ]:
# 7c. lgbm_full_d — route + small interactions
def build_route_small():
    tr_enr, te_enr = build_route_features(tr, te)
    tr_enr = _add_small_features(tr_enr)
    te_enr = _add_small_features(te_enr)
    feats = [c for c in tr_enr.columns if c not in ('id','satisfaction','__y__','__src__')]
    Xtr = tr_enr[feats].copy()
    Xte = te_enr[feats].copy()
    for c in CAT_COLS:
        if c in Xtr.columns:
            Xtr[c] = Xtr[c].astype('category')
            Xte[c] = Xte[c].astype('category')
    Xtr, Xte = make_aligned_categoricals(Xtr, Xte)
    return Xtr, Xte

Xtr_d, Xte_d = build_route_small()
oof_d, test_d = lgbm_5fold(Xtr_d, y, Xte_d, folds, seed=SEED, params=CHAMPION_BASE_PARAMS)
print(f'lgbm_full_d OOF AUC: {roc_auc_score(y, oof_d):.6f}')
np.save(os.path.join(OOF_DIR, 'lgbm_full_d.npy'), oof_d.astype(np.float32))
np.save(os.path.join(PRED_DIR, 'lgbm_full_d.npy'), test_d.astype(np.float32))

## 8. Stacking — nested-CV logistic stacker on logits

Greedy hill-climb: start from the best single model, then keep adding
members (with replacement) that raise the blend AUC at any C value in
{0.005, 0.01, 0.02, 0.05, 0.1, 0.2}.

In [ ]:
from sklearn.linear_model import LogisticRegression

def _to_logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def stack_lr(oofs, tests, y, C=0.005, n_repeats=3, seed=SEED):
    n = len(y)
    oof_stack = np.zeros(n)
    for rep in range(n_repeats):
        skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed+rep)
        oof_rep = np.zeros(n)
        for tr, va in skf2.split(np.zeros(n), y):
            Xtr = np.column_stack([_to_logit(o[tr]) for o in oofs])
            Xva = np.column_stack([_to_logit(o[va]) for o in oofs])
            lr = LogisticRegression(C=C, solver='lbfgs', max_iter=200)
            lr.fit(Xtr, y[tr])
            oof_rep[va] = lr.predict_proba(Xva)[:, 1]
        oof_stack += oof_rep / n_repeats
    Xt = np.column_stack([_to_logit(t) for t in tests])
    Xall = np.column_stack([_to_logit(o) for o in oofs])
    lr = LogisticRegression(C=C, solver='lbfgs', max_iter=400)
    lr.fit(Xall, y)
    test_stack = lr.predict_proba(Xt)[:, 1]
    return oof_stack, test_stack

def best_blend(min_names=2, exclude=None):
    names = sorted([os.path.basename(p)[:-4] for p in glob.glob(os.path.join(OOF_DIR, '*.npy'))
                    if os.path.basename(p)[:-4] not in ('stack',)])
    if exclude:
        names = [n for n in names if n not in exclude]
    if len(names) < min_names:
        return None
    oofs = {n: np.load(os.path.join(OOF_DIR, f'{n}.npy')) for n in names}
    tests = {n: np.load(os.path.join(PRED_DIR, f'{n}.npy')) for n in names}
    aucs = {n: roc_auc_score(y, oofs[n]) for n in names}
    seed_name = max(names, key=lambda n: aucs[n])
    sel = [seed_name]; cur_auc = aucs[seed_name]
    improved = True
    while improved:
        improved = False
        for n in names:
            cand = sel + [n]
            best_C = None; best_a = cur_auc
            for C in [0.005, 0.01, 0.02, 0.05, 0.1, 0.2]:
                os_, ts_ = stack_lr([oofs[k] for k in cand], [tests[k] for k in cand], y, C=C, n_repeats=2)
                a = roc_auc_score(y, os_)
                if a > best_a + 1e-5:
                    best_a = a; best_C = C; best_o = os_; best_t = ts_
            if best_C is not None:
                sel = cand; cur_auc = best_a; improved = True
    # Final stack at best C
    best_C = None; best_a = cur_auc; best_o = None; best_t = None
    for C in [0.005, 0.01, 0.02, 0.05, 0.1, 0.2]:
        os_, ts_ = stack_lr([oofs[k] for k in sel], [tests[k] for k in sel], y, C=C, n_repeats=3)
        a = roc_auc_score(y, os_)
        if a > best_a + 1e-5 or best_C is None:
            if a > best_a:
                best_a = a
            best_C = C; best_o = os_; best_t = ts_
    return f'stack({"+".join(sel)})_C={best_C}', best_a, best_o, best_t

name, auc, oof_stack, test_stack = best_blend()
print(f'Best blend: {name}')
print(f'OOF AUC: {auc:.6f}')
np.save(os.path.join(OOF_DIR, 'stack.npy'), oof_stack.astype(np.float32))
np.save(os.path.join(PRED_DIR, 'stack.npy'), test_stack.astype(np.float32))

## 9. Submission

Write the blend to `submissions/best.csv` and the champion single to
`submissions/best_single.csv`.

In [ ]:
samp = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

def write_submission(probs, path, name=''):
    sub = pd.DataFrame({'id': samp['id'].values, 'satisfaction': probs})
    assert len(sub) == len(samp)
    assert (sub['id'].values == samp['id'].values).all()
    assert sub['satisfaction'].notna().all()
    assert sub['satisfaction'].between(0, 1).all()
    sub.to_csv(path, index=False)
    print(f'  wrote {path}  ({name})  min={sub["satisfaction"].min():.4f} '
          f'max={sub["satisfaction"].max():.4f}  mean={sub["satisfaction"].mean():.4f}')

write_submission(test_stack, os.path.join(SUB_DIR, 'best.csv'), name='best (blend)')
write_submission(test_g,      os.path.join(SUB_DIR, 'best_single.csv'), name='best_single (lgbm_full_g)')

## 10. Per-segment diagnostic (optional)

Quick sanity check on the weakest segments. The 'Personal Travel' cohort is
the single largest weakness, regardless of class.

In [ ]:
def per_segment_auc(oof, y, mask):
    return roc_auc_score(y[mask], oof[mask])

print('1D segments on lgbm_full_g OOF:')
for col in ('Type of Travel', 'Class', 'Customer Type'):
    for v in tr[col].unique():
        m = (tr[col] == v).values
        if m.sum() < 1000:
            continue
        print(f'  {col}={v:>15}  n={m.sum():>7}  AUC={per_segment_auc(oof_g, y, m):.4f}')

print('\n3D weakest segments (Class × Type × Customer):')
groups = tr.groupby(['Class', 'Type of Travel', 'Customer Type'])
rows = []
for k, sub in groups:
    if len(sub) < 500:
        continue
    m = sub.index.values
    rows.append((k, len(sub), per_segment_auc(oof_g, y, m)))
for k, n, a in sorted(rows, key=lambda r: r[2])[:5]:
    print(f'  {str(k):>60}  n={n:>7}  AUC={a:.4f}')

## Summary

**Current best (OOF AUC):**
- Single champion: `lgbm_full_g` = **0.960757** (3-seed average of Optuna params)
- 5-OOF blend (lgbm_full_g + cat_d4 + lgbm_full + lgbm_full_te + lgbm_raw): **0.960865**

**Submission files:**
- `submissions/best.csv` — blend, upload to Kaggle for best LB
- `submissions/best_single.csv` — single champion, less correlated risk

**What worked:**
1. Route group features (+0.0016 over raw LGBM)
2. Optuna LGBM hyperparameter search (+0.0003 over default LGBM)
3. 3-seed averaging at Optuna params (+0.0001)
4. CatBoost adds ordered-boosting diversity (BLEND MEMBER)
5. Greedy hill-climb stacker on logit space

**What didn't work:**
- XGBoost (couldn't converge at any depth/lr tried)
- 4-way target encoding (rejected, but still a blend member)
- Small hand-engineered features (rejected, but still a blend member)

**Known weaknesses:**
- 'Personal Travel' segment AUC = 0.83 (vs 0.96 for Business)
- Eco × Personal × Loyal (n=185k) is the largest weak segment